# 2. Predict hit versus non-hit

We compare ten valid classifier families. The original empty VotingClassifier is excluded because it has no component estimators. A majority-class baseline shows why overall accuracy can be misleading.

In [1]:
from pipeline_lib import RunContext, train_models
ctx = RunContext.from_environment("classification")
print("Profile:", ctx.config["profile"])
print("Artifacts:", ctx.run_dir)
from IPython.display import Image, display
train, holdout = ctx.data()
print('Training class counts:')
display((train.popularity > ctx.config['hit_threshold']).value_counts())

Profile: full
Artifacts: /Users/anishech/Projects/ML_Spotify/runs/20261021_140544_769703_full
Training class counts:


popularity
False    55731
True      1825
Name: count, dtype: int64

Each candidate is a pipeline: genre target encoding → standard scaling → PCA (95% variance) → classifier. All preprocessing is fitted within each training fold; genre encoding also cross-fits its training values. Models are selected by positive-class F1 in cross-validation, not holdout accuracy. Linear/tree models use class weights where supported. Search budgets come from the selected JSON configuration.

In [ ]:
classification_results = train_models(ctx, classification=True)
display(classification_results)

In [ ]:
import json
selection = json.loads(ctx.path('metrics', 'classification_selection.json').read_text())
print('Chosen using training CV:', selection['model'])
display(Image(filename=str(ctx.path('plots', 'classification_confusion_matrix.png'))))

Read hit precision (how many shortlisted songs are hits), hit recall (how many real hits are found), F1 and average precision alongside accuracy. Confusion matrix rows are actual labels; columns are predicted labels. Every saved Joblib artifact contains all required preprocessing.